In [ ]:
"""
=====================================================================
MBG Tweet Classification - Multi-Model Hierarchical Ensemble
Models   : IndoBERT + XLM-RoBERTa  (extensible ke model lain)
Pipeline : 1 Seed → Holdout Split → 5-Fold CV per model
           → Hierarchical Ensemble (intra-model fold → inter-model)
           → Softmax-Temperature Weighting
           → Holdout Eval → Threshold Tuning
Metric   : Balanced Accuracy

Install  :
  pip install transformers torch openpyxl scikit-learn scipy
=====================================================================
"""

import pandas as pd
import numpy as np
import re
import torch
import random
import warnings
warnings.filterwarnings("ignore")

from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy.special import softmax as scipy_softmax


# ╔══════════════════════════════════════════════════════════╗
# ║                       CONFIG                             ║
# ╚══════════════════════════════════════════════════════════╝

MODEL_CONFIGS = [
    {
        "name":       "indolem/indobert-base-uncased",
        "label":      "IndoBERT",
        "max_len":    80,
        "lr":         2e-5,
        "batch_size": 64,
    },
    {
        "name":       "xlm-roberta-base",
        "label":      "XLM-R",
        "max_len":    72,
        "lr":         1e-5,
        "batch_size": 64,
    },
]

EPOCHS               = 100
WARMUP_RATIO         = 0.1
WEIGHT_DECAY         = 0.01
PATIENCE             = 3
N_FOLDS              = 5
HOLDOUT_RATIO        = 0.2
SEED                 = 42
GRAD_ACCUM_STEPS     = 1       # naikkan jika OOM (misal 2 atau 4)
USE_AMP              = True    # mixed precision (otomatis off jika CPU)
USE_WEIGHTED_ENSEMBLE = True

# Softmax-temperature weighting:
# Makin tinggi T → bobot makin terkonsentrasi ke model/fold terbaik.
# T=1 ≈ softmax biasa | T=8 ≈ tajam tapi tidak collapse | T=∞ → argmax
ENSEMBLE_TEMPERATURE = 8.0

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = USE_AMP and torch.cuda.is_available()

DATA_DIR = "/kaggle/input/datasets/muhammadalabrar12/dataset-mbg"


# ╔══════════════════════════════════════════════════════════╗
# ║                   REPRODUCIBILITY                        ║
# ╚══════════════════════════════════════════════════════════╝

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ╔══════════════════════════════════════════════════════════╗
# ║                  TEXT PREPROCESSING                      ║
# ║                                                          ║
# ║  Order operasi:                                          ║
# ║  1. Emoji → teks  (sebelum lower, sebelum regex clean)  ║
# ║  2. Lowercase                                            ║
# ║  3. URL / mention / hashtag / karakter cleanup           ║
# ║  4. Slang normalization (setelah token bersih)           ║
# ╚══════════════════════════════════════════════════════════╝

# ── Emoji → padanan teks Bahasa Indonesia ─────────────────
# Emoji harus dikonversi SEBELUM regex [^\w\s@#] yang akan menghapusnya.
EMOJI_MAP: dict[str, str] = {
    "😂": "lucu",    "🤣": "lucu",
    "😭": "sedih",   "😢": "sedih",
    "😡": "marah",   "🤬": "marah",    "😤": "marah",
    "❤️": "cinta",   "🥰": "cinta",    "😍": "cinta",
    "👍": "bagus",   "🙏": "mohon",
    "😊": "senang",  "😁": "senang",   "🎉": "senang",
    "😱": "kaget",   "😨": "takut",
    "🤔": "bingung", "😴": "ngantuk",
}

# ── Slang → bentuk baku ───────────────────────────────────
# Diaplikasikan per-token (bukan substring) agar tidak menimpa
# kata yang sudah benar. Contoh: "yang" tidak berubah menjadi "yangang".
SLANG_MAP: dict[str, str] = {
    # Negasi
    "gk":    "tidak",   "ga":    "tidak",   "nggak": "tidak",
    "ngga":  "tidak",   "gak":   "tidak",   "ndak":  "tidak",
    # Intensifier
    "bgt":   "banget",  "bngt":  "banget",
    # Konjungsi / kata fungsi
    "yg":    "yang",    "tp":    "tapi",    "tpi":   "tapi",
    "dgn":   "dengan",  "dg":    "dengan",
    "utk":   "untuk",   "krn":   "karena",  "karna": "karena",
    "udh":   "sudah",   "udah":  "sudah",   "sdh":   "sudah",
    "lg":    "lagi",    "lgi":   "lagi",
    "jg":    "juga",    "jga":   "juga",
    "sy":    "saya",    "aq":    "saya",    "gw":    "saya",
    "gue":   "saya",    "lo":    "kamu",    "lu":    "kamu",
    "km":    "kamu",    "bkn":   "bukan",   "blm":   "belum",
    "skrg":  "sekarang","skrang": "sekarang",
    "emg":   "memang",  "emang": "memang",
    "aja":   "saja",    "doang": "saja",
    "klo":   "kalau",   "kalo":  "kalau",
}


def _replace_emoji(text: str) -> str:
    """Ganti emoji ke padanan teks Bahasa Indonesia."""
    for emoji, word in EMOJI_MAP.items():
        text = text.replace(emoji, f" {word} ")
    return text


def _normalize_slang(text: str) -> str:
    """Ganti slang per-token agar tidak menimpa substring."""
    tokens = text.split()
    return " ".join(SLANG_MAP.get(tok, tok) for tok in tokens)


def preprocess(text: str) -> str:
    text = str(text)
    text = _replace_emoji(text)               # 1. emoji → teks (sebelum lowercase)
    text = text.lower()                       # 2. lowercase
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " @user ", text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[^\w\s@#]", " ", text)   # 3. buang karakter aneh
    text = re.sub(r"\s+", " ", text).strip()
    text = _normalize_slang(text)             # 4. slang setelah token bersih
    return text


# ╔══════════════════════════════════════════════════════════╗
# ║                    DATASET CLASS                         ║
# ╚══════════════════════════════════════════════════════════╝

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def make_loader(texts, labels, tokenizer, max_len, batch_size, shuffle) -> DataLoader:
    ds = TweetDataset(texts, labels, tokenizer, max_len)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=2, pin_memory=torch.cuda.is_available(),
    )


# ╔══════════════════════════════════════════════════════════╗
# ║          TRAIN EPOCH  (dengan AMP + grad accum)          ║
# ╚══════════════════════════════════════════════════════════╝

def train_epoch(
    model, loader, optimizer, scheduler, loss_fn, scaler
) -> tuple[float, float]:
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    optimizer.zero_grad()

    for step, batch in enumerate(loader, start=1):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        with autocast(enabled=USE_AMP):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss   = loss_fn(logits, labels) / GRAD_ACCUM_STEPS

        if USE_AMP:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            if USE_AMP:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        preds_all.extend(logits.argmax(dim=-1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

    return total_loss / len(loader), balanced_accuracy_score(labels_all, preds_all)


# ╔══════════════════════════════════════════════════════════╗
# ║                     EVAL EPOCH                           ║
# ╚══════════════════════════════════════════════════════════╝

def eval_epoch(
    model, loader, loss_fn
) -> tuple[float, float, np.ndarray, np.ndarray]:
    model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            with autocast(enabled=USE_AMP):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss   = loss_fn(logits, labels)

            total_loss  += loss.item()
            all_logits.append(logits.cpu().float().numpy())
            all_labels.append(labels.cpu().numpy())

    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    ba         = balanced_accuracy_score(all_labels, all_logits.argmax(axis=1))

    return total_loss / len(loader), ba, all_logits, all_labels


# ╔══════════════════════════════════════════════════════════╗
# ║                   PREDICT LOGITS                         ║
# ╚══════════════════════════════════════════════════════════╝

def predict_logits(model, loader) -> np.ndarray:
    model.eval()
    all_logits = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with autocast(enabled=USE_AMP):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            all_logits.append(logits.cpu().float().numpy())

    return np.concatenate(all_logits, axis=0)


# ╔══════════════════════════════════════════════════════════╗
# ║                   TRAIN ONE FOLD                         ║
# ╚══════════════════════════════════════════════════════════╝

def train_one_fold(
    model_cfg:      dict,
    X_tr:           list,
    y_tr:           list,
    X_val:          list,
    y_val:          list,
    tokenizer,
    loss_fn,
    num_classes:    int,
    model_path:     str,
    holdout_loader: DataLoader,
    test_loader:    DataLoader,
    fold_idx:       int,
) -> tuple[float, np.ndarray, np.ndarray]:
    """
    Melatih satu model pada satu fold dengan early stopping.

    Return:
        best_val_ba    : float
        holdout_logits : (N_holdout, num_classes)
        test_logits    : (N_test,    num_classes)
    """
    train_loader = make_loader(
        X_tr, y_tr, tokenizer, model_cfg["max_len"],
        model_cfg["batch_size"], shuffle=True,
    )
    val_loader = make_loader(
        X_val, y_val, tokenizer, model_cfg["max_len"],
        model_cfg["batch_size"], shuffle=False,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_cfg["name"], num_labels=num_classes
    ).to(DEVICE)

    total_steps  = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    optimizer    = AdamW(model.parameters(), lr=model_cfg["lr"], weight_decay=WEIGHT_DECAY)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
    )
    scaler = GradScaler(enabled=USE_AMP)

    best_val_ba      = 0.0
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_ba                       = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler)
        va_loss, va_ba, va_logits, va_labels = eval_epoch(model, val_loader, loss_fn)

        print(f"    Epoch {epoch:>3}/{EPOCHS} | "
              f"Train Loss={tr_loss:.4f} BA={tr_ba:.4f} | "
              f"Val   Loss={va_loss:.4f} BA={va_ba:.4f}")

        if va_ba > best_val_ba:
            best_val_ba      = va_ba
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
            print(f"      ✅ Best saved → val BA={best_val_ba:.4f}")
        else:
            patience_counter += 1
            print(f"      ⏳ No improvement ({patience_counter}/{PATIENCE})")
            if patience_counter >= PATIENCE:
                print(f"      🛑 Early stopping at epoch {epoch}")
                break

    # Inference dari checkpoint terbaik.
    # Holdout tidak pernah masuk loop training/val → tidak ada leakage.
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    holdout_logits = predict_logits(model, holdout_loader)
    test_logits    = predict_logits(model, test_loader)

    del model
    torch.cuda.empty_cache()

    return best_val_ba, holdout_logits, test_logits


# ╔══════════════════════════════════════════════════════════╗
# ║                   ENSEMBLE UTILITIES                     ║
# ╚══════════════════════════════════════════════════════════╝

def normalize_weights(
    val_bas: list[float],
    temperature: float = ENSEMBLE_TEMPERATURE,
) -> np.ndarray:
    """
    Softmax-temperature weighting.

      w_i = softmax(val_BA_i × T)
          = exp(val_BA_i × T) / Σ exp(val_BA_j × T)

    Makin tinggi T → bobot makin terkonsentrasi ke model/fold terbaik.
    Stabilisasi numerik: kurangi max sebelum exp agar tidak overflow.
    """
    arr = np.array(val_bas, dtype=np.float64)
    arr = arr - arr.max()                    # numerical stability
    exp_arr = np.exp(arr * temperature)
    return exp_arr / exp_arr.sum()


def ensemble_logits(
    logits_list: list[np.ndarray],
    weights: np.ndarray | None = None,
) -> np.ndarray:
    """
    Weighted (atau simple) average dari sekumpulan logit arrays.

    logits_list : list of (N, C)
    weights     : (M,) normalized, atau None → simple mean
    Return      : (N, C)
    """
    stacked = np.stack(logits_list, axis=0)          # (M, N, C)
    if weights is None:
        return stacked.mean(axis=0)
    w = weights[:, np.newaxis, np.newaxis]            # (M, 1, 1)
    return (stacked * w).sum(axis=0)                  # (N, C)


def _intra_model_ensemble(
    holdout_logits: list[np.ndarray],
    test_logits:    list[np.ndarray],
    val_bas:        list[float],
) -> tuple[np.ndarray, np.ndarray, float]:
    """
    Level 1 — intra-model: gabungkan N fold dari satu model → satu logits.

    Bobot fold : softmax-temperature dari val_BA masing-masing fold.

    Return:
        ens_holdout : (N_holdout, C)
        ens_test    : (N_test,    C)
        mean_val_ba : float — representasi kualitas model untuk Level 2
    """
    fold_weights = normalize_weights(val_bas)
    ens_holdout  = ensemble_logits(holdout_logits, fold_weights)
    ens_test     = ensemble_logits(test_logits,    fold_weights)
    mean_val_ba  = float(np.mean(val_bas))
    return ens_holdout, ens_test, mean_val_ba


# ╔══════════════════════════════════════════════════════════╗
# ║               THRESHOLD TUNING (MULTICLASS)              ║
# ╚══════════════════════════════════════════════════════════╝

def tune_confidence_threshold(
    logits:         np.ndarray,
    true_labels:    np.ndarray,
    fallback_class: int,
    thresholds:     np.ndarray | None = None,
) -> tuple[float, float, float]:
    """
    Jika max_prob dari softmax < threshold → assign ke fallback_class.
    Sweep threshold dan return yang menghasilkan BA tertinggi.

    Return: best_ba, best_threshold, baseline_ba
    """
    probs       = scipy_softmax(logits, axis=1)
    baseline_ba = balanced_accuracy_score(true_labels, probs.argmax(axis=1))

    if thresholds is None:
        thresholds = np.arange(0.0, 0.95, 0.01)

    best_ba, best_thr = baseline_ba, 0.0

    for thr in thresholds:
        preds                              = probs.argmax(axis=1).copy()
        preds[probs.max(axis=1) < thr]    = fallback_class
        ba = balanced_accuracy_score(true_labels, preds)
        if ba > best_ba:
            best_ba, best_thr = ba, thr

    return best_ba, best_thr, baseline_ba


# ══════════════════════════════════════════════════════════
#                          MAIN
# ══════════════════════════════════════════════════════════

set_seed(SEED)

model_labels_list = [cfg["label"] for cfg in MODEL_CONFIGS]
print(f"Device       : {DEVICE}  (AMP={'on' if USE_AMP else 'off'})")
print(f"Seed         : {SEED}")
print(f"Models       : {model_labels_list}")
print(f"Folds        : {N_FOLDS}  →  total checkpoints = {N_FOLDS * len(MODEL_CONFIGS)}")
print(f"Holdout      : {int(HOLDOUT_RATIO * 100)}% of train")
print(f"Ens. Temp.   : {ENSEMBLE_TEMPERATURE}\n")


# ── 1. Load & preprocess ──────────────────────────────────

train_df = pd.read_excel(f"{DATA_DIR}/train.xlsx")
test_df  = pd.read_excel(f"{DATA_DIR}/test.xlsx")
sub_df   = pd.read_excel(f"{DATA_DIR}/submission.xlsx")

train_df["clean_text"] = train_df["full_text"].apply(preprocess)
test_df["clean_text"]  = test_df["full_text"].apply(preprocess)


# ── 2. Label encoding ─────────────────────────────────────

le = LabelEncoder()
train_df["label_enc"] = le.fit_transform(train_df["label"])
num_classes = len(le.classes_)

print(f"Classes ({num_classes}): {le.classes_}")
print(f"Distribution:\n{train_df['label'].value_counts()}\n")

fallback_class = int(np.where(le.classes_ == "Lainnya")[0][0]) \
    if "Lainnya" in le.classes_ else 0

X_full = train_df["clean_text"].tolist()
y_full = np.array(train_df["label_enc"].tolist())


# ── 3. Holdout split ──────────────────────────────────────
# Dilakukan SEKALI sebelum apapun. Holdout dikunci dari training.

X_kfold, X_holdout, y_kfold, y_holdout = train_test_split(
    X_full, y_full,
    test_size=HOLDOUT_RATIO,
    stratify=y_full,
    random_state=SEED,
)
print(f"K-Fold pool : {len(X_kfold)} samples")
print(f"Holdout     : {len(X_holdout)} samples  ← dikunci dari training\n")


# ── 4. Class weights (dari K-Fold pool saja) ──────────────

class_counts = np.bincount(y_kfold, minlength=num_classes)
cw_tensor    = torch.tensor(1.0 / (class_counts / class_counts.sum()), dtype=torch.float).to(DEVICE)
cw_tensor    = cw_tensor / cw_tensor.sum() * num_classes
loss_fn      = torch.nn.CrossEntropyLoss(weight=cw_tensor)


# ── 5. Pre-load semua tokenizer ───────────────────────────

print("Loading tokenizers...")
for cfg in MODEL_CONFIGS:
    cfg["tokenizer"] = AutoTokenizer.from_pretrained(cfg["name"])
    print(f"  ✓ {cfg['label']} tokenizer loaded")
print()

# Holdout & test loader dibuat per-model (tiap tokenizer berbeda)
for cfg in MODEL_CONFIGS:
    cfg["holdout_loader"] = make_loader(
        X_holdout, y_holdout.tolist(),
        cfg["tokenizer"], cfg["max_len"], cfg["batch_size"], shuffle=False,
    )
    cfg["test_loader"] = make_loader(
        test_df["clean_text"].tolist(), None,
        cfg["tokenizer"], cfg["max_len"], cfg["batch_size"], shuffle=False,
    )


# ── 6. Multi-model × K-Fold training ──────────────────────

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

model_results = {
    cfg["label"]: {"holdout_logits": [], "test_logits": [], "val_bas": []}
    for cfg in MODEL_CONFIGS
}

for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_kfold, y_kfold), start=1):
    X_tr  = [X_kfold[i] for i in tr_idx]
    y_tr  = y_kfold[tr_idx].tolist()
    X_val = [X_kfold[i] for i in va_idx]
    y_val = y_kfold[va_idx].tolist()

    for cfg in MODEL_CONFIGS:
        print(f"\n{'═'*65}")
        print(f"  MODEL: {cfg['label']}  |  FOLD {fold_idx}/{N_FOLDS}")
        print(f"{'═'*65}")

        model_path = f"best_{cfg['label']}_fold{fold_idx}.pt"

        best_val_ba, holdout_logits, test_logits = train_one_fold(
            model_cfg      = cfg,
            X_tr           = X_tr,
            y_tr           = y_tr,
            X_val          = X_val,
            y_val          = y_val,
            tokenizer      = cfg["tokenizer"],
            loss_fn        = loss_fn,
            num_classes    = num_classes,
            model_path     = model_path,
            holdout_loader = cfg["holdout_loader"],
            test_loader    = cfg["test_loader"],
            fold_idx       = fold_idx,
        )

        model_results[cfg["label"]]["holdout_logits"].append(holdout_logits)
        model_results[cfg["label"]]["test_logits"].append(test_logits)
        model_results[cfg["label"]]["val_bas"].append(best_val_ba)

        print(f"\n  {cfg['label']} | Fold {fold_idx} → val BA = {best_val_ba:.4f}")


# ── 7. Per-model fold summary ─────────────────────────────

print(f"\n{'═'*65}")
print("  PER-MODEL FOLD SUMMARY")
print(f"{'═'*65}")
for cfg in MODEL_CONFIGS:
    lbl  = cfg["label"]
    bas  = model_results[lbl]["val_bas"]
    fstr = "  ".join([f"F{i+1}={v:.4f}" for i, v in enumerate(bas)])
    print(f"  {lbl:<12} | Mean={np.mean(bas):.4f} ± {np.std(bas):.4f} | {fstr}")


# ── 8. Hierarchical Ensemble ──────────────────────────────
#
# LEVEL 1 — intra-model (per-fold → per-model):
#   Bobot  : softmax-temperature dari val_BA tiap fold
#   Output : 1 pasang logits per model
#
# LEVEL 2 — inter-model (per-model → final):
#   Bobot  : softmax-temperature dari mean_val_BA tiap model
#   Output : 1 pasang logits final

print(f"\n{'═'*65}")
print("  HIERARCHICAL ENSEMBLE")
print(f"  Temperature = {ENSEMBLE_TEMPERATURE}")
print(f"{'═'*65}")

level1_holdout: dict[str, np.ndarray] = {}
level1_test:    dict[str, np.ndarray] = {}
level1_mean_ba: dict[str, float]      = {}

# ── Level 1: intra-model ──
for cfg in MODEL_CONFIGS:
    lbl = cfg["label"]
    ens_h, ens_t, mean_ba = _intra_model_ensemble(
        holdout_logits = model_results[lbl]["holdout_logits"],
        test_logits    = model_results[lbl]["test_logits"],
        val_bas        = model_results[lbl]["val_bas"],
    )
    level1_holdout[lbl] = ens_h
    level1_test[lbl]    = ens_t
    level1_mean_ba[lbl] = mean_ba

    l1_ba    = balanced_accuracy_score(y_holdout, ens_h.argmax(axis=1))
    fold_w   = normalize_weights(model_results[lbl]["val_bas"])

    print(f"\n  [{lbl}]  Level-1 holdout BA = {l1_ba:.4f}  (mean val_BA={mean_ba:.4f})")
    for i, (vba, fw) in enumerate(zip(model_results[lbl]["val_bas"], fold_w), start=1):
        print(f"    Fold {i}: val_BA={vba:.4f}  weight={fw:.4f}")

# ── Level 2: inter-model ──
model_labels_ordered = [cfg["label"] for cfg in MODEL_CONFIGS]
model_mean_bas       = [level1_mean_ba[lbl] for lbl in model_labels_ordered]
model_weights        = normalize_weights(model_mean_bas)

final_holdout_logits = ensemble_logits(
    [level1_holdout[lbl] for lbl in model_labels_ordered],
    model_weights,
)
final_test_logits = ensemble_logits(
    [level1_test[lbl] for lbl in model_labels_ordered],
    model_weights,
)

final_holdout_ba = balanced_accuracy_score(y_holdout, final_holdout_logits.argmax(axis=1))

print(f"\n  [Inter-model Level-2]")
for lbl, mba, mw in zip(model_labels_ordered, model_mean_bas, model_weights):
    print(f"    {lbl:<12}: mean_val_BA={mba:.4f}  weight={mw:.4f}")
print(f"\n  Final holdout BA (before threshold) = {final_holdout_ba:.4f}")

# Untuk summary final
per_model_holdout_ba = {
    lbl: balanced_accuracy_score(y_holdout, level1_holdout[lbl].argmax(axis=1))
    for lbl in model_labels_ordered
}
ensemble_label = f"Hierarchical (T={ENSEMBLE_TEMPERATURE})"


# ── 9. Threshold tuning ───────────────────────────────────

print(f"\n{'═'*65}")
print("  THRESHOLD TUNING  (multiclass confidence-based)")
print(f"  Fallback class : '{le.classes_[fallback_class]}' (idx={fallback_class})")
print(f"{'═'*65}")

best_ba, best_thr, baseline_ba = tune_confidence_threshold(
    final_holdout_logits, y_holdout, fallback_class,
)


# ── 10. Final test prediction ─────────────────────────────

test_probs = scipy_softmax(final_test_logits, axis=1)
test_preds = test_probs.argmax(axis=1).copy()
if best_thr > 0.0:
    test_preds[test_probs.max(axis=1) < best_thr] = fallback_class
y_pred_labels = le.inverse_transform(test_preds)


# ── 11. Final summary ─────────────────────────────────────

print(f"\n{'═'*65}")
print("  FINAL SUMMARY")
print(f"{'═'*65}")

print(f"\n  ── Per-model holdout BA (Level-1 intra-model ensemble) ──")
for lbl, ba in per_model_holdout_ba.items():
    print(f"    {lbl:<12} : {ba:.4f}")

print(f"\n  ── Cross-model holdout BA (Level-2 inter-model ensemble) ──")
print(f"    {ensemble_label} : {final_holdout_ba:.4f}")

print(f"\n  ── Model weight ranking (Level-2) ──")
ranked_models = sorted(
    zip(model_labels_ordered, model_mean_bas, model_weights),
    key=lambda x: -x[1],
)
print(f"  {'Model':<14} {'Mean val_BA':>12} {'Weight':>8}")
print(f"  {'─'*36}")
for lbl, mba, mw in ranked_models:
    print(f"  {lbl:<14} {mba:>12.4f} {mw:>8.4f}")

print(f"\n  ── Threshold tuning ──")
print(f"    Holdout BA (no threshold)    : {baseline_ba:.4f}")
delta = best_ba - baseline_ba
arrow = "↑" if delta > 0 else "→"
print(f"    Holdout BA (thr={best_thr:.2f})       : {best_ba:.4f}  ({arrow} {abs(delta):.4f})")

print(f"\n  Seed        : {SEED}")
print(f"  Folds       : {N_FOLDS}")
print(f"  Models      : {model_labels_list}")
print(f"  Temperature : {ENSEMBLE_TEMPERATURE}")


# ── 12. Save submission ───────────────────────────────────

sub_df["label"] = y_pred_labels
out_file = "submission_hierarchical_ensemble.xlsx"
sub_df.to_excel(out_file, index=False)

print(f"\nPrediction distribution:\n{pd.Series(y_pred_labels).value_counts()}")
print(f"\n✅ Saved : {out_file}")
print(f"   Final holdout BA : {best_ba:.4f}  @ threshold={best_thr:.2f}")